# 07 — Model Monitoring Simulation

Simulates production monitoring by splitting users into time-cohort buckets based on `latest_transaction_date`, training on the earliest cohort, and scoring later cohorts.

Tracks AUC, top-decile recall, and Population Stability Index (PSI) on prediction probability across cohorts.

Persists `analytics.mart_model_monitoring`.

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score

sns.set_theme(style='whitegrid')
DUCKDB_PATH = Path('../data/processed/kkbox.duckdb')
RANDOM_STATE = 42

In [ ]:
con = duckdb.connect(str(DUCKDB_PATH))
df = con.execute('SELECT * FROM analytics.mart_user_churn_features').df()
df['latest_transaction_date'] = pd.to_datetime(df['latest_transaction_date'])
df.dropna(subset=['latest_transaction_date'], inplace=True)
df.sort_values('latest_transaction_date', inplace=True)
print('Date range:', df['latest_transaction_date'].min(), '->', df['latest_transaction_date'].max())
print('Total users:', len(df))

## Cohort split

Quartiles of `latest_transaction_date` so each cohort has roughly equal size.

In [ ]:
df['cohort'] = pd.qcut(df['latest_transaction_date'].rank(method='first'),
                        q=4, labels=['Q1_train', 'Q2_holdout1', 'Q3_holdout2', 'Q4_holdout3'])
df.groupby('cohort').agg(n=('msno', 'count'),
                          start=('latest_transaction_date', 'min'),
                          end=('latest_transaction_date', 'max'),
                          churn_rate=('is_churn', 'mean'))

In [ ]:
TARGET = 'is_churn'
DROP = ['msno', 'is_churn', 'registration_init_time', 'latest_transaction_date',
         'latest_expire_date', 'cohort']
categorical = ['gender', 'age_band', 'registered_via', 'latest_payment_method_id']
numerical = [c for c in df.columns if c not in DROP + categorical]

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_enc = df.copy()
df_enc[categorical] = encoder.fit_transform(df[categorical].astype(str))

## Train on Q1, score later cohorts

In [ ]:
train = df_enc[df_enc['cohort'] == 'Q1_train']
X_train = train[numerical + categorical]
y_train = train[TARGET].astype(int).values
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical)
params = dict(objective='binary', metric='binary_logloss', learning_rate=0.05,
               num_leaves=63, min_data_in_leaf=200, scale_pos_weight=scale_pos_weight,
               verbose=-1, random_state=RANDOM_STATE)
model = lgb.train(params, lgb_train, num_boost_round=400)

In [ ]:
def psi(expected, actual, bins=10):
    breakpoints = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(breakpoints) < 3:
        return np.nan
    e, _ = np.histogram(expected, bins=breakpoints)
    a, _ = np.histogram(actual,   bins=breakpoints)
    e = e / max(e.sum(), 1)
    a = a / max(a.sum(), 1)
    e = np.where(e == 0, 1e-6, e)
    a = np.where(a == 0, 1e-6, a)
    return float(np.sum((a - e) * np.log(a / e)))

def top_decile_recall(y_true, scores):
    s = pd.DataFrame({'y': y_true, 'p': scores}).sort_values('p', ascending=False)
    n = len(s)
    top_n = max(int(0.1 * n), 1)
    captured = s['y'].iloc[:top_n].sum()
    return float(captured / max(s['y'].sum(), 1))

train_scores = model.predict(X_train)
rows = []
for c in ['Q1_train', 'Q2_holdout1', 'Q3_holdout2', 'Q4_holdout3']:
    sub = df_enc[df_enc['cohort'] == c]
    if len(sub) == 0:
        continue
    X = sub[numerical + categorical]
    y = sub[TARGET].astype(int).values
    p = model.predict(X)
    auc = roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan
    recall10 = top_decile_recall(y, p)
    rows.append({
        'cohort':        str(c),
        'cohort_start':  sub['cohort'].astype(str).iloc[0],
        'n_users':       len(sub),
        'churn_rate':    float(y.mean()),
        'auc':           auc,
        'recall_top_decile': recall10,
        'psi_vs_train':  psi(train_scores, p),
        'mean_pred':     float(p.mean()),
        'std_pred':      float(p.std()),
    })
monitoring = pd.DataFrame(rows)
monitoring

## Visualise drift

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
monitoring.plot(x='cohort', y='auc', kind='line', marker='o', ax=axes[0], color='#1f77b4', legend=False)
axes[0].set_title('AUC across cohorts'); axes[0].set_ylim(0.5, 1.0)
monitoring.plot(x='cohort', y='recall_top_decile', kind='line', marker='o', ax=axes[1], color='#ff7f0e', legend=False)
axes[1].set_title('Top-decile recall')
monitoring.plot(x='cohort', y='psi_vs_train', kind='bar', ax=axes[2], color='#d62728', legend=False)
axes[2].axhline(0.10, color='gray', linestyle='--', label='warn 0.10')
axes[2].axhline(0.20, color='black', linestyle='--', label='alert 0.20')
axes[2].set_title('PSI of predicted probability')
axes[2].legend()
plt.tight_layout()
plt.show()

## Persist

In [ ]:
con.register('mon_df', monitoring)
con.execute('CREATE OR REPLACE TABLE analytics.mart_model_monitoring AS SELECT * FROM mon_df')
print('Wrote analytics.mart_model_monitoring:', len(monitoring), 'rows')
con.close()

## Interpretation

Compare AUC and top-decile recall across cohorts. If they decay sharply from Q1 to Q4, the model is degrading and would need retraining in production. The PSI bars indicate input drift: > 0.10 = warn, > 0.20 = alert.

Even if metrics are stable here (the v2 snapshot is short), this notebook proves the analytical setup needed to monitor a production model.